# 02 — Modelagem, avaliação e interpretação

Continua a análise de `AnaliseExploratoria.ipynb` (T03–T11) sem alterá-la. Este notebook reproduz o split e os modelos (mesmos parâmetros, `random_state=42`) e cobre:

1. Análise de correlação (T07)
2. Avaliação no conjunto de teste (T12)
3. Validação cruzada
4. Feature importance e SHAP (T13, T14)
5. Discussão crítica de uso prático (T15)

Classe positiva: **M (maligno)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, make_scorer)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

df = pd.read_csv("../data/data.csv").drop(columns=['id', 'Unnamed: 32'])
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

## T07 — Análise de correlação

In [ ]:
corr = X.corr()
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().dropna().rename('correlação').reset_index()
             .rename(columns={'level_0': 'feature A', 'level_1': 'feature B'}))
pares['|r|'] = pares['correlação'].abs()
top_pares = pares.sort_values('|r|', ascending=False).head(15).drop(columns='|r|')
print(f"Pares com |r| > 0,9: {(pares['|r|'] > 0.9).sum()} de {len(pares)}")
top_pares.round(3).reset_index(drop=True)

In [ ]:
y_bin = (y == 'M').astype(int)
corr_alvo = X.corrwith(y_bin).sort_values(key=np.abs, ascending=False)
corr_alvo.head(10).round(3)

In [ ]:
plt.figure(figsize=(7, 4))
corr_alvo.sort_values().tail(10).plot.barh(title="Correlação com o alvo (M=1) — top 10")
plt.tight_layout()
plt.savefig("../reports/figures/correlacao_alvo.png", dpi=150)
plt.show()

**Conclusão T07**

**Correlação**
- 21 dos 435 pares de features têm |r| > 0,9. Os mais altos são `radius`, `perimeter` e `area`, que medem o tamanho do núcleo: `radius_mean` × `perimeter_mean` = 0,998, e há a mesma redundância entre as versões `mean` e `worst` (`perimeter_mean` × `perimeter_worst` = 0,970).
- As features mais associadas ao diagnóstico são `concave points_worst` (0,79), `perimeter_worst` (0,78), `concave points_mean` (0,78) e `radius_worst` (0,78), as mesmas apontadas por SHAP e importância no notebook 03. Ou seja, os dois métodos independentes concordam.
- **Consequência prática:** a informação de "tamanho" está repetida em até 9 colunas. Isso não prejudica a acurácia dos modelos, mas explica os coeficientes instáveis da Regressão Logística e a importância dispersa por permutação. Reduzir dimensionalidade (PCA) ou remover features redundantes é uma melhoria possível, não aplicada aqui para preservar o pipeline do T06.


## Split e modelos (mesmos parâmetros dos T08–T11)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42)
print(len(X_train), len(X_val), len(X_test))

def make_model(classifier):
    pre = ColumnTransformer(transformers=[('num', Pipeline(steps=[('scaler', StandardScaler())]), list(X.columns))])
    return Pipeline(steps=[('preprocessor', pre), ('classifier', classifier)])

modelos = {
    'Regressão Logística': make_model(LogisticRegression(random_state=42)),
    'Random Forest': make_model(RandomForestClassifier(random_state=42)),
    'KNN': make_model(KNeighborsClassifier()),
}
for m in modelos.values():
    m.fit(X_train, y_train)

## Conferência com os resultados do T09–T11

In [ ]:
esperado = {'Regressão Logística': 0.977, 'Random Forest': 0.965, 'KNN': 0.977}
for nome, m in modelos.items():
    acc = accuracy_score(y_val, m.predict(X_val))
    assert round(acc, 3) == esperado[nome], (nome, acc)
    print(f"{nome:20s} acurácia (validação) = {acc:.3f}  OK")

## T12 — Avaliação no conjunto de teste

O teste só é usado aqui, uma única vez, com os modelos treinados apenas no treino. A classe positiva é **M (maligno)**: o recall dela mede quantos casos malignos o modelo detecta; um falso negativo (maligno classificado como benigno) é o erro clinicamente mais grave.

In [ ]:
linhas = []
matrizes = {}
for nome, m in modelos.items():
    pred = m.predict(X_test)
    linhas.append({
        'Modelo': nome,
        'Accuracy': accuracy_score(y_test, pred),
        'Recall (M)': recall_score(y_test, pred, pos_label='M'),
        'Precisão (M)': precision_score(y_test, pred, pos_label='M'),
        'F1 (M)': f1_score(y_test, pred, pos_label='M'),
    })
    matrizes[nome] = confusion_matrix(y_test, pred, labels=['B', 'M'])

resultados = pd.DataFrame(linhas).set_index('Modelo').round(3)
resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nome, cm) in zip(axes, matrizes.items()):
    ConfusionMatrixDisplay(cm, display_labels=['Benigno', 'Maligno']).plot(ax=ax, colorbar=False)
    ax.set_title(nome)
plt.tight_layout()
plt.savefig("../reports/figures/matrizes_confusao_teste.png", dpi=150)
plt.show()

for nome, cm in matrizes.items():
    tn, fp, fn, tp = cm.ravel()
    print(f"{nome:20s} falsos negativos (M como B) = {fn} | falsos positivos (B como M) = {fp}")

**Conclusão T12** (teste: 86 amostras, 32 malignas):

| Modelo | Accuracy | Recall (M) | Precisão (M) | F1 (M) |
|---|---|---|---|---|
| Regressão Logística | 0,988 | 0,969 | 1,000 | 0,984 |
| Random Forest | 0,965 | 0,906 | 1,000 | 0,951 |
| KNN | 0,953 | 0,906 | 0,967 | 0,935 |

- **Métrica principal: recall da classe maligna.** Um falso negativo deixa passar um tumor maligno, erro mais grave que um falso positivo, que apenas leva a um exame adicional. A acurácia sozinha esconde isso, pois a base tem ~63% de benignos.
- A Regressão Logística teve o melhor resultado em todas as métricas: 1 falso negativo e nenhum falso positivo. Random Forest e KNN erraram 3 casos malignos cada.
- **Cautela:** com apenas 32 casos malignos no teste, cada erro muda o recall em ~3 pontos percentuais; as diferenças entre modelos (1 a 3 erros) não são estatisticamente conclusivas. Validação cruzada daria uma estimativa mais estável.
- O modelo apoia a triagem; a decisão diagnóstica é sempre do médico.


## Robustez — validação cruzada estratificada

Usa apenas treino + validação (85% dos dados); o conjunto de teste não é tocado. Mostra a média e o desvio das métricas em 10 folds, para avaliar a estabilidade dos resultados do T12.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'recall_M': 'recall', 'f1_M': 'f1'}
y_temp_bin = (y_temp == 'M').astype(int)   # M=1 para o scorer usar pos_label=1

linhas = []
for nome, m in modelos.items():
    r = cross_validate(m, X_temp, y_temp_bin, cv=cv, scoring=scoring)
    linhas.append({
        'Modelo': nome,
        'Accuracy': f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
        'Recall (M)': f"{r['test_recall_M'].mean():.3f} ± {r['test_recall_M'].std():.3f}",
        'F1 (M)': f"{r['test_f1_M'].mean():.3f} ± {r['test_f1_M'].std():.3f}",
    })
pd.DataFrame(linhas).set_index('Modelo')

**Conclusão da validação cruzada**

(10 folds, treino + validação)

| Modelo | Accuracy | Recall (M) | F1 (M) |
|---|---|---|---|
| Regressão Logística | 0,973 ± 0,023 | 0,956 ± 0,042 | 0,963 ± 0,031 |
| Random Forest | 0,956 ± 0,024 | 0,928 ± 0,043 | 0,941 ± 0,031 |
| KNN | 0,965 ± 0,021 | 0,911 ± 0,044 | 0,950 ± 0,030 |

- A Regressão Logística fica à frente nas três métricas, como no teste do T12, o que dá mais confiança de que a ordenação não foi acaso do split.
- Ainda assim, o desvio de ~0,04 no recall é do tamanho da diferença entre modelos (0,956 vs 0,928): a vantagem sobre o Random Forest é sugestiva, não conclusiva. O KNN tem o menor recall médio (0,911), o pior para triagem, apesar de acurácia parecida.


## T13 e T14 — Interpretabilidade

In [ ]:
modelo_lr = modelos['Regressão Logística']
modelo_rf = modelos['Random Forest']

## T13 — Feature importance

### Random Forest: importância por impureza

Calculada no treino; pode favorecer features correlacionadas e contínuas, por isso é comparada com a importância por permutação.

In [ ]:
imp_rf = pd.Series(modelo_rf.named_steps['classifier'].feature_importances_, index=X.columns).sort_values()
imp_rf.tail(10).plot.barh(figsize=(7, 4), title="Random Forest — importância por impureza (top 10)")
plt.tight_layout()
plt.savefig("../reports/figures/importancia_rf_impureza.png", dpi=150)
plt.show()
imp_rf.sort_values(ascending=False).head(10).round(3)

### Importância por permutação (conjunto de validação)

Queda do recall de M ao embaralhar cada feature. Usa a validação para não tocar o teste.

In [ ]:
scorer = make_scorer(recall_score, pos_label='M')
perm = permutation_importance(modelo_rf, X_val, y_val, scoring=scorer, n_repeats=30, random_state=42)
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values()
imp_perm.tail(10).plot.barh(figsize=(7, 4), title="Random Forest — importância por permutação (recall de M)")
plt.tight_layout()
plt.savefig("../reports/figures/importancia_rf_permutacao.png", dpi=150)
plt.show()
imp_perm.sort_values(ascending=False).head(10).round(4)

### Regressão Logística: coeficientes (features padronizadas)

In [ ]:
coef = pd.Series(modelo_lr.named_steps['classifier'].coef_[0], index=X.columns)
assert list(modelo_lr.named_steps['classifier'].classes_) == ['B', 'M']  # coef positivo => aumenta P(maligno)
top = coef.reindex(coef.abs().sort_values().index).tail(10)
top.plot.barh(figsize=(7, 4), title="Regressão Logística — coeficientes (top 10 por |valor|)")
plt.tight_layout()
plt.savefig("../reports/figures/coeficientes_lr.png", dpi=150)
plt.show()
top.sort_values(ascending=False).round(3)

## T14 — SHAP Explica a contribuição de cada feature para a probabilidade/log-odds de **maligno**, nas amostras do teste (nenhuma decisão de modelo é tomada com base nisso).

In [ ]:
def dados_transformados(modelo, Xd):
    pre = modelo.named_steps['preprocessor']
    return pd.DataFrame(pre.transform(Xd), columns=X.columns, index=Xd.index)

# Regressão Logística (linear)
Xtr_lr = dados_transformados(modelo_lr, X_train)
Xte_lr = dados_transformados(modelo_lr, X_test)
expl_lr = shap.LinearExplainer(modelo_lr.named_steps['classifier'], Xtr_lr)
sv_lr = expl_lr(Xte_lr)

shap.summary_plot(sv_lr.values, Xte_lr, show=False)
plt.title("SHAP — Regressão Logística (teste)")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_lr.png", dpi=150)
plt.show()

In [ ]:
# Random Forest (árvores)
Xte_rf = dados_transformados(modelo_rf, X_test)
expl_rf = shap.TreeExplainer(modelo_rf.named_steps['classifier'])
sv_rf = expl_rf(Xte_rf)
idx_m = list(modelo_rf.named_steps['classifier'].classes_).index('M')
vals_rf = sv_rf.values[:, :, idx_m] if sv_rf.values.ndim == 3 else sv_rf.values

shap.summary_plot(vals_rf, Xte_rf, show=False)
plt.title("SHAP — Random Forest (teste, classe M)")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_rf.png", dpi=150)
plt.show()

In [ ]:
# Importância global média |SHAP| dos dois modelos
comp = pd.DataFrame({
    'LR': np.abs(sv_lr.values).mean(axis=0),
    'RF': np.abs(vals_rf).mean(axis=0),
}, index=X.columns)
comp['LR'] /= comp['LR'].sum(); comp['RF'] /= comp['RF'].sum()
comp.sort_values('RF', ascending=False).head(10).round(3)

## Conclusões T13 e T14

- **Random Forest:** as features mais importantes, tanto por impureza quanto por SHAP, são as medidas `worst` de tamanho e concavidade: `area_worst`, `concave points_worst`, `perimeter_worst` e `radius_worst`. No SHAP, valores altos delas aumentam a probabilidade de maligno e valores baixos a reduzem.
- **Multicolinearidade:** `radius`, `perimeter` e `area` são quase a mesma informação. Por isso a importância por permutação é baixa e dispersa, ao embaralhar uma delas, as correlatas compensam. Importâncias individuais dessas features não devem ser lidas isoladamente, e sim como um grupo "tamanho do núcleo".
- **Regressão Logística:** o SHAP da regressão é mais distribuído entre as features do que o do Random Forest. Os coeficientes servem para o modelo, mas não devem ser interpretados como efeito individual de cada variável.
- **Uso no contexto clínico:** SHAP permite mostrar ao médico quais características levaram a cada predição, mas explica o modelo e não a doença: é correlação aprendida nesta base, sem garantia de causalidade. A decisão continua sendo do médico.


## T15 — Discussão crítica de uso prático

**Papel do modelo.** O modelo é uma ferramenta de apoio à triagem: sinaliza casos que merecem atenção prioritária, mas não emite diagnóstico. O médico integra o resultado com histórico, exames de imagem e biópsia, e tem a palavra final.

**Qual erro importa mais.** No teste, a Regressão Logística errou 1 dos 32 casos malignos (recall 0,969) e não gerou falsos positivos; Random Forest e KNN deixaram passar 3 malignos cada (recall 0,906). Um falso negativo atrasa o diagnóstico de um câncer; um falso positivo gera um exame adicional e ansiedade. Por isso o recall da classe maligna é a métrica de referência, e o modelo com maior recall (Regressão Logística) é o candidato mais adequado. Se o hospital quiser reduzir ainda mais os falsos negativos, o limiar de decisão pode ser rebaixado (por exemplo, sinalizar como suspeito com probabilidade de maligno menor que 0,5), aceitando mais falsos positivos. Esse ajuste não foi feito neste trabalho e exigiria definir com a equipe clínica quanto de falso positivo é aceitável.

**Interpretabilidade.** SHAP mostra que o modelo se apoia em tamanho e concavidade do núcleo (`area_worst`, `concave points_worst`, `perimeter_worst`), coerente com o conhecimento clínico e com a correlação dessas features com o diagnóstico. Isso permite ao médico conferir se a justificativa faz sentido. Porém é a explicação do modelo, não da doença: são associações aprendidas nesta base, sem garantia de causalidade.

**Limitações.**
- Base pequena (569 casos) e de uma única origem (Breast Cancer Wisconsin, features extraídas de imagens digitalizadas de punção aspirativa). Nada garante o mesmo desempenho em outra população, equipamento ou protocolo de coleta.
- Poucos casos malignos no teste (32): as métricas têm intervalos largos, como mostra a validação cruzada (±0,04 no recall).
- Features altamente redundantes (T07); o desempenho vem de um sinal forte e simples, o que pode não se manter em casos ambíguos ou raros.
- Não foi feita calibração de probabilidades nem análise por subgrupos.

**Antes de qualquer uso real:** validação externa em dados de outro serviço, estudo prospectivo comparando a triagem com e sem o modelo, aprovação regulatória aplicável a software de apoio à decisão clínica, monitoramento de desempenho ao longo do tempo e tratamento adequado dos dados de pacientes (LGPD). Este trabalho usa somente dado público e é um exercício acadêmico.
